# Synthetic Pools Accuracy (DeepFLR)
**Summary:** Evaluates PTM localization accuracy comparing Prosit-PTM, MSFragger, and DeepFLR on synthetic pools.
Computes Precision-Recall metrics and FLR on Target/Decoy datasets.

**Required Files:**
- Prosit rescore results (`rescore.percolator_with_prob.psms.txt`)
- MSFragger search results (`psm.tsv`)
- DeepFLR results (`mono_target_decoy_msms_samplemodelmonomz_modelresult_MSF_IT.csv`, `mono_target_decoy_msms_sample_msf_IT.csv`)
- Ground truth pool datasets (`pool{1..5}.csv`)


In [ ]:
import numpy as np
import pandas as pd
import re
import warnings
import matplotlib.pyplot as plt
import matplotlib
import os

from scipy.interpolate import interp1d, UnivariateSpline
from tqdm import tqdm
from glob import iglob

import spectrum_fundamentals.constants as c
from spectrum_fundamentals.metrics.percolator import Percolator
from spectrum_fundamentals.mod_string import internal_without_mods

warnings.filterwarnings('ignore')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


## Configuration
Define file paths and modification mappings.


In [ ]:
PROSIT_RESCORE_PATH = '<PATH_TO_PROSIT_RESCORE>'
MSFRAGGER_PSM_PATH = '<PATH_TO_MSFRAGGER_PSM>'
DEEPFLR_MODEL_RESULT_PATH = '<PATH_TO_DEEPFLR_MODEL_RESULT>'
DEEPFLR_SEQUENCE_PATH = '<PATH_TO_DEEPFLR_SEQUENCE>'
POOL_CSV_PATH_TEMPLATE = '<PATH_TO_POOL_TEMPLATE>'
PLOT_OUTPUT_DIR = '<PATH_TO_PLOT_OUTPUT>'
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)


In [ ]:
MSFRAGGER_VAR_MODS = {
    "C\\[160\\]": "C[UNIMOD:4]",
    "M\\[147\\]": "M[UNIMOD:35]",
    "K\\[230\\]": "K[UNIMOD:737]",
    "K\\[305\\]": "K[UNIMOD:2016]",
    "K\\[214\\]": "K[UNIMOD:214]",
    "R\\[157\\]": "R[UNIMOD:7]",
    "Q\\[129\\]": "Q[UNIMOD:7]",
    "N\\[115\\]": "N[UNIMOD:7]",
    "n\\[230\\]": "[UNIMOD:737]-",
    "n\\[305\\]": "[UNIMOD:2016]-",
    "n\\[214\\]": "[UNIMOD:214]-",
    "n\\[43\\]": "[UNIMOD:1]-",
    "S\\[167\\]": "S[UNIMOD:21]",
    "T\\[181\\]": "T[UNIMOD:21]",
    "Y\\[243\\]": "Y[UNIMOD:21]",
    "H\\[217\\]": "H[UNIMOD:21]",
    "C(?!\\[)": "C[UNIMOD:4]",
    "C\\[119\\]": "C[UNIMOD:35]",
    "C\\[117\\]": "C[UNIMOD:34]",
    "C\\[222\\]": "C[UNIMOD:312]",
    "H\\[153\\]": "H[UNIMOD:35]",
    "H\\[151\\]": "H[UNIMOD:34]",
    "K\\[144\\]": "K[UNIMOD:35]",
    "K\\[142\\]": "K[UNIMOD:34]",
    "K\\[156\\]": "K[UNIMOD:36]",
    "K\\[185\\]": "K[UNIMOD:4]",
    "K\\[242\\]": "K[UNIMOD:1848]",
    "K\\[511\\]": "K[UNIMOD:535]",
    "K\\[170.10\\]": "K[UNIMOD:1]",
    "K\\[471\\]": "K[UNIMOD:1293]",
    "K\\[458\\]": "K[UNIMOD:1990]",
    "P\\[113\\]": "P[UNIMOD:35]",
    "W\\[202\\]": "W[UNIMOD:35]",
    "D\\[129\\]": "D[UNIMOD:34]",
    "E\\[143\\]": "E[UNIMOD:34]",
    "I\\[127\\]": "I[UNIMOD:34]",
    "L\\[127\\]": "L[UNIMOD:34]",
    "N\\[128\\]": "N[UNIMOD:34]",
    "R\\[170\\]": "R[UNIMOD:34]",
    "K\\[184\\]": "K[UNIMOD:58]",
    "K\\[187\\]": "K[UNIMOD:5634]",  # Acetyl_label+monomethyl
    "K\\[173\\]": "K[UNIMOD:56]",  # Heavy_Aceytl
    "K\\[198\\]": "K[UNIMOD:1289]",
    "n\\[57\\]": "[UNIMOD:58]-",
    "n\\[60\\]": "[UNIMOD:59]-",
    "n\\[120\\]": "[UNIMOD:411]-",
    "K\\[170.14\\]": "K[UNIMOD:37]",
}


## Helper Functions


In [ ]:
def add_phospho(row):
    seq = row['Phospopeptide sequence']
    positions = row['positions']
    for pos in positions:
        seq = seq[:int(pos)] + '[UNIMOD:21]' + seq[int(pos):]
    return seq

def find_start_end_index(found_pep,expected_pep):
    for match in re.finditer(found_pep, expected_pep):
        return pd.Series([match.start()+1, match.end()+1])

def generate_possible_phospho(unmod_peptide,positions,start_index,end_index):
    all_posibble_peptides = []
    current_mod_peptide = ''
    for pos in positions:
        if int(pos)<start_index or int(pos)>end_index:
            continue
        else:
            if current_mod_peptide !='':
                all_posibble_peptides.append(current_mod_peptide[:int(pos)-start_index+1] + '[UNIMOD:21]' + current_mod_peptide[int(pos)-start_index+1:])
            current_mod_peptide = unmod_peptide[:int(pos)-start_index+1] + '[UNIMOD:21]' + unmod_peptide[int(pos)-start_index+1:]
            all_posibble_peptides.append(current_mod_peptide)
    return all_posibble_peptides

def count_STY(sequence):
    return sequence.count('S') + sequence.count('T') + sequence.count('Y')

All_prosit_df = All_prosit_df[All_prosit_df['q-value']<0.01]

All_fragger_df['delta_score_temp'] = All_fragger_df['STY:79.9663'].apply(lambda x: sorted([float(num) for num in re.findall(r'\((0\.\d+)\)', x)], reverse=True))


## Load and Preprocess Data
Load Prosit, MSFragger, and DeepFLR results.


In [ ]:
psms_df = pd.read_csv(PROSIT_RESCORE_PATH, sep='\t')
psms_df['peptide'] = psms_df['peptide'].str.replace('_.','')
psms_df['peptide'] = psms_df['peptide'].str.replace('._','')
psms_df['peptide'] = psms_df['peptide'].str.replace('[UNIMOD:35]','')
psms_df['peptide'] = psms_df['peptide'].str.replace('[UNIMOD:4]','')
psms_df['peptide'] = psms_df['peptide'].str.replace('[UNIMOD:1]-','')
psms_df['unmod_peptide'] = internal_without_mods(psms_df['peptide'].values)

psms_fragger_df = pd.read_csv(MSFRAGGER_PSM_PATH, sep='\t')
psms_fragger_df.replace({"Modified Peptide": MSFRAGGER_VAR_MODS}, regex=True, inplace=True)
psms_fragger_df = psms_fragger_df.dropna(subset=['STY:79.9663'])
psms_fragger_df = psms_fragger_df[psms_fragger_df['Modified Peptide'].str.contains('UNIMOD:21')]
psms_fragger_df['Modified Peptide'] = psms_fragger_df['Modified Peptide'].str.replace('[UNIMOD:35]','')
psms_fragger_df['Modified Peptide'] = psms_fragger_df['Modified Peptide'].str.replace('[UNIMOD:4]','')
psms_fragger_df['Modified Peptide'] = psms_fragger_df['Modified Peptide'].str.replace('[UNIMOD:1]-','')

psms_fragger_df['sty_count'] = psms_fragger_df.apply(lambda x: count_STY(x['Modified Peptide']),axis=1)
psms_fragger_df['ph_count'] = psms_fragger_df.apply(lambda x: x['Modified Peptide'].count('21'),axis=1)
psms_df['sty_count'] = psms_df.apply(lambda x: count_STY(x['peptide']),axis=1)
psms_df['ph_count'] = psms_df.apply(lambda x: x['peptide'].count('21'),axis=1)

psms_df = psms_df[psms_df['sty_count']>psms_df['ph_count']]
psms_fragger_df = psms_fragger_df[psms_fragger_df['sty_count']>psms_fragger_df['ph_count']]

# Load DeepFLR
df=pd.read_csv(DEEPFLR_MODEL_RESULT_PATH)
df["score"]=df["score"].str.replace("tensor(",'',regex=False).str.replace(".)",'',regex=False).str.replace(")",'',regex=False).astype("float")
df1=pd.read_csv(DEEPFLR_SEQUENCE_PATH)
df1.columns=["SourceFile","Fspectrum","PP.Charge","exp_strip_sequence","Peptide","key_x"]
df=pd.merge(df,df1,on=["SourceFile","Fspectrum","key_x","PP.Charge"],how="left")
df=df[["SourceFile","Fspectrum","Peptide","PEP.StrippedSequence","PP.Charge","key_x","score"]]
df["striptrue"]=df["Peptide"].str.replace("1",'',regex=False).str.replace("2",'',regex=False).str.replace("3",'',regex=False).str.replace("4",'',regex=False)

df=df.sort_values(by='score',ascending=False).reset_index(drop=True)
dfmax=df.drop_duplicates(subset=['SourceFile', 'Fspectrum', 'Peptide'])
dftruemax=dfmax.loc[dfmax["striptrue"]==dfmax["PEP.StrippedSequence"]]
dffalsemax=dfmax.loc[dfmax["striptrue"]!=dfmax["PEP.StrippedSequence"]]
dftrue=df.loc[df["striptrue"]==df["PEP.StrippedSequence"]]

df=pd.concat([dfmax,dftrue]).drop_duplicates()
f =lambda x: x.score.iloc[0]-x.score.iloc[1]
dfdelta=df.groupby(["Fspectrum","SourceFile","Peptide"]).apply(f)
dfdelta=pd.DataFrame(dfdelta).reset_index(drop=False)
dfdelta.columns=['Fspectrum','SourceFile',"Peptide",'deltascore']
zz=pd.merge(dfdelta,dffalsemax,on=['Fspectrum','SourceFile',"Peptide"],how='right')
ww=pd.merge(dfdelta,dftruemax,on=['Fspectrum','SourceFile',"Peptide"],how='right')
df=pd.concat([zz,ww]).drop_duplicates()
df=df.loc[df["striptrue"]==df["PEP.StrippedSequence"]]

deep_flr=df
deep_flr['Peptide'] = deep_flr['key_x'].str.replace('2','').str.replace('3','').str.replace('4','')
deep_flr['Peptide'] = deep_flr['Peptide'].str.replace('1','[UNIMOD:21]')
deep_flr['unmod_peptide'] = internal_without_mods(deep_flr['Peptide'].values)


## Map to Synthetic Pools


In [ ]:
All_prosit_pools, All_fragger_pools, All_flr_pools, expected_sequences = [], [], [], []
total_pep, total_double_mod, total_adj_mod = 0, 0, 0

for j in range(1,6):
    expected_seq = pd.read_csv(POOL_CSV_PATH_TEMPLATE.format(j))
    if j ==3:
        expected_seq['positions'] = expected_seq['modified position in peptide'].apply(lambda x: [x])
    else:
        expected_seq['positions'] = expected_seq['modified position in peptide'].apply(lambda x: x.split('&'))
    total_pep += len(expected_seq)
    for i in range(len(expected_seq)):
        expected_seq['positions'].values[i].reverse()
    expected_seq['mod_seq'] = expected_seq.apply(lambda x: add_phospho(x),axis=1)
    expected_seq['peptide_id'] = ["Peptide_" + str(i + 1) for i in range(len(expected_seq))]
    expected_sequences.append(expected_seq)
    
    total_double_mod += len(expected_seq[expected_seq['# of sites in peptide']==2])
    pattern = r"(?:(S|T|Y)(?:\[UNIMOD:21\])?(S|T|Y)\[UNIMOD:21\]|(S|T|Y)\[UNIMOD:21\](S|T|Y)(?!\[UNIMOD:21\]))"
    expected_seq['adj'] = expected_seq["mod_seq"].str.contains(pattern, regex=True)
    total_adj_mod+= len(expected_seq[expected_seq["mod_seq"].str.contains(pattern, regex=True)])

    prosit_pool_df = psms_df[psms_df['filename'].str.contains('pool'+str(j)+'_')]
    prosit_pool_df = prosit_pool_df.merge(expected_seq,left_on='unmod_peptide',right_on='Phospopeptide sequence',how='inner')
    prosit_pool_df['correct_loc'] = prosit_pool_df.apply(lambda x: x['peptide']==x['mod_seq'],axis=1)
    All_prosit_pools.append(prosit_pool_df)

    fragger_pool_df = psms_fragger_df[psms_fragger_df['Spectrum File'].str.contains('pool'+str(j)+'_')]
    fragger_pool_df = fragger_pool_df.merge(expected_seq,left_on='Peptide',right_on='Phospopeptide sequence',how='inner')
    fragger_pool_df['correct_loc'] = fragger_pool_df.apply(lambda x: x['Modified Peptide']==x['mod_seq'],axis=1)
    fragger_pool_df = fragger_pool_df[~fragger_pool_df['Modified Peptide'].isna()]
    fragger_pool_df = fragger_pool_df[fragger_pool_df['Modified Peptide'].str.contains('21')]
    All_fragger_pools.append(fragger_pool_df)

    deepflr_pool_df = deep_flr[deep_flr['SourceFile'].str.contains('pool'+str(j)+'_')]
    deepflr_pool_df = deepflr_pool_df.merge(expected_seq,left_on='unmod_peptide',right_on='Phospopeptide sequence',how='inner')
    deepflr_pool_df['correct_loc'] = deepflr_pool_df.apply(lambda x: x['Peptide']==x['mod_seq'],axis=1)
    deepflr_pool_df = deepflr_pool_df[deepflr_pool_df['Peptide'].str.contains('21')]
    All_flr_pools.append(deepflr_pool_df)

All_prosit_df = pd.concat(All_prosit_pools)
All_fragger_df = pd.concat(All_fragger_pools)
All_flr_df = pd.concat(All_flr_pools)
expected_sequences = pd.concat(expected_sequences)

All_prosit_df['count_phospho'] = All_prosit_df['peptide'].apply(lambda x: x.count('UNIMOD:21') )
All_fragger_df['count_phospho'] = All_fragger_df['Modified Peptide'].apply(lambda x: x.count('UNIMOD:21') )
All_flr_df['count_phospho'] = All_flr_df['Peptide'].apply(lambda x: x.count('UNIMOD:21') )

All_prosit_df = All_prosit_df[All_prosit_df['# of sites in peptide']==All_prosit_df['count_phospho']]
All_fragger_df = All_fragger_df[All_fragger_df['# of sites in peptide']==All_fragger_df['count_phospho']]
All_flr_df = All_flr_df[All_flr_df['# of sites in peptide']==All_flr_df['count_phospho']]


## Filter and Compute Metrics


In [ ]:
All_prosit_pools_fil = All_prosit_df.sort_values('loc_prob',ascending=False)
All_prosit_pools_fil.drop_duplicates('peptide',inplace=True)
All_fragger_pools_fil_df = All_fragger_df.sort_values('STY:79.9663 Best Localization',ascending=False)
All_fragger_pools_fil_df.drop_duplicates('Modified Peptide',inplace=True)

All_flr_pools_fil = All_flr_df.sort_values('deltascore',ascending=False)
All_flr_pools_fil.drop_duplicates('Peptide',inplace=True)

All_fragger_df['filename'] = All_fragger_df['Spectrum File'].apply(lambda x: x.split('\\')[-1].split('.')[0].replace('interact-',''))
All_fragger_df['ScanNr']= All_fragger_df['Spectrum'].apply(lambda x: int(x.split('.')[1]))
All_prosit_df['ScanNr'] = All_prosit_df[['PSMId','filename']].apply(lambda x: int(x['PSMId'][len(x['filename']):].split('-')[1]),axis=1)


All_prosit_pools_msf_fil_df = All_prosit_df.merge(All_fragger_df[['filename','ScanNr']],on=['filename','ScanNr'],how='inner')
All_prosit_pools_msf_fil_df.drop_duplicates(['filename','ScanNr'],inplace=True)

All_prosit_df = All_prosit_df[~((All_prosit_df['peptide_id']=='Peptide_24') & (All_prosit_df['PSMId'].str.contains('pool2')))]

All_prosit_df = All_prosit_df[~((All_prosit_df['peptide_id']=='Peptide_27') & (All_prosit_df['PSMId'].str.contains('pool5')))]


All_prosit_df = All_prosit_df[All_prosit_df['q-value']<0.01]

All_fragger_df['delta_score_temp'] = All_fragger_df['STY:79.9663'].apply(lambda x: sorted([float(num) for num in re.findall(r'\((0\.\d+)\)', x)], reverse=True))


All_fragger_df['delta_score'] = All_fragger_df.apply(lambda x: x['delta_score_temp'][0] - x['delta_score_temp'][1] if x['count_phospho']==1
                                                     else x['delta_score_temp'][0] - x['delta_score_temp'][2],axis=1)

All_fragger_df['filename_scan'] = All_fragger_df[['filename','ScanNr']].apply(lambda x: x['filename']+str(x['ScanNr']),axis=1)

All_prosit_df['filename_scan'] = All_prosit_df[['filename','ScanNr']].apply(lambda x: x['filename']+str(x['ScanNr']),axis=1)

filename_scan_drop = All_fragger_df.loc[((All_fragger_df['delta_score']==0) & (All_fragger_df['correct_loc'])),['filename_scan']]

All_prosit_df = All_prosit_df[~All_prosit_df['filename_scan'].isin(filename_scan_drop.values.flatten())]

All_flr_df['filename_scan'] = All_flr_df[['SourceFile','Fspectrum']].apply(lambda x: x['SourceFile']+str(x['Fspectrum']),axis=1)
All_fragger_df = All_fragger_df[All_fragger_df['filename_scan'].isin(All_flr_df['filename_scan'].values)]

All_prosit_df['loc_prob_delta'] =  All_prosit_df['loc_prob_delta'].apply(lambda x: round(x,2))
All_prosit_df['loc_prob'] =  All_prosit_df['loc_prob'].apply(lambda x: round(x,2))


## Evaluate Adjustments


In [ ]:
def get_flr_recall(df):
    sorted_labels = df['correct_loc'].values
    cumulative_decoy_count = np.cumsum(sorted_labels == False) 
    cumulative_target_count= np.cumsum(sorted_labels == True)
    ldr = Percolator.fdrs_to_qvals(cumulative_decoy_count / (cumulative_target_count+ cumulative_decoy_count))
    flr = Percolator.fdrs_to_qvals(cumulative_decoy_count / (cumulative_target_count))
    precision = 1 - ldr
    return flr, precision, cumulative_target_count, cumulative_target_count[-1]

All_prosit_pools_non_adj_df =  All_prosit_df[(~All_prosit_df['adj'])]
All_prosit_df_adj = All_prosit_df[(All_prosit_df['adj'])]
All_fragger_pools_non_adj_df =  All_fragger_df[(~All_fragger_df['adj'])]
All_fragger_df_adj = All_fragger_df[(All_fragger_df['adj'])]
All_flr_pools_non_adj_df =  All_flr_df[(~All_flr_df['adj'])]
All_flr_df_adj = All_flr_df[(All_flr_df['adj'])]

flr_prosit, precision_prosit, cumulative_target_count_prosit, recall_prosit = get_flr_recall(All_prosit_pools_non_adj_df)
flr_prosit_adj, precision_prosit_adj, cumulative_target_count_prosit_adj, recall_prosit_adj = get_flr_recall(All_prosit_df_adj)
flr_fragger, precision_fragger, cumulative_target_count_fragger, recall_fragger = get_flr_recall(All_fragger_pools_non_adj_df)
flr_fragger_adj, precision_fragger_adj, cumulative_target_count_fragger_adj, recall_fragger_adj = get_flr_recall(All_fragger_df_adj)
flr_flr, precision_flr, cumulative_target_count_flr, recall_flr = get_flr_recall(All_flr_pools_non_adj_df)
flr_flr_adj, precision_flr_adj, cumulative_target_count_flr_adj, recall_flr_adj = get_flr_recall(All_flr_df_adj)

def find_longest_decreasing_streak(arr):
    if len(arr) == 0:
        return []
    streaks = []
    current_streak = []
    for i in range(1, len(arr)):
        if arr[i] < arr[i - 1]:
            current_streak.append(i)
        else:
            if current_streak:
                streaks.append(current_streak)
                current_streak = []
    if current_streak:
        streaks.append(current_streak)
    if not streaks:
        return []
    return max(streaks, key=len)


## Plotting


In [ ]:
All_fragger_df['delta_score'] = All_fragger_df.apply(lambda x: x['delta_score_temp'][0] - x['delta_score_temp'][1] if x['count_phospho']==1
                                                     else x['delta_score_temp'][0] - x['delta_score_temp'][2],axis=1)

All_prosit_df['filename_scan'] = All_prosit_df[['filename','ScanNr']].apply(lambda x: x['filename']+str(x['ScanNr']),axis=1)


In [ ]:
# Plotting the two Precision-Recall curves
plt.figure(figsize=(8, 6))


cumulative_target_smooth,flr_smooth = fit_curve(cumulative_target_count_prosit,flr_prosit,0.1)
plt.plot(flr_smooth, cumulative_target_smooth,label='Prosit-PTM', color='royalblue', linewidth=2)



# Curve 2 - Light Blue
cumulative_target_ad_smooth,flr_adj_smooth = fit_curve(cumulative_target_count_prosit_adj,flr_prosit_adj)

plt.plot(flr_adj_smooth, cumulative_target_ad_smooth,label='Prosit-PTM_adj', color='limegreen', linewidth=2)

# Curve 3 - Green



cumulative_target_fragger_smooth,flr_fragger_smooth = fit_curve(cumulative_target_count_fragger,flr_fragger)

plt.plot(flr_fragger_smooth, cumulative_target_fragger_smooth, label='MSFragger', color='royalblue', linewidth=2,linestyle='--')


# Curve 4 - Light Green


cumulative_target_fragger_adj_smooth,flr_fragger_adj_smooth = fit_curve(cumulative_target_count_fragger_adj,flr_fragger_adj)


plt.plot(flr_fragger_adj_smooth, cumulative_target_fragger_adj_smooth, label='MSFragger_adj', color='limegreen', linewidth=2,linestyle='--')


# Curve 3 - Green
cumulative_target_flr_smooth,flr_flr_smooth = fit_curve(cumulative_target_count_flr,flr_flr)

plt.plot(flr_flr_smooth, cumulative_target_flr_smooth, label='DeepFLR', color='royalblue', linewidth=2,linestyle=':')

# Curve 4 - Light Green

cumulative_target_flr_adj_smooth,flr_flr_adj_smooth = fit_curve(cumulative_target_count_flr_adj,flr_flr_adj)
remove_ind = find_longest_decreasing_streak(flr_flr_adj_smooth)
flr_flr_adj_smooth = np.delete(flr_flr_adj_smooth,remove_ind  +  list(range(111,129)))
cumulative_target_flr_adj_smooth = np.delete(cumulative_target_flr_adj_smooth,  remove_ind   +  list(range(111,129)))


plt.plot(flr_flr_adj_smooth, cumulative_target_flr_adj_smooth, label='DeepFLR_adj', color='limegreen', linewidth=2,linestyle=':')
plt.axvline(x=0.011, color='r', linestyle='--', label='FLR=0.01')
plt.axvline(x=0.05, color='r', linestyle='--', label='FLR=0.05')

plt.xlim(0,0.1)
plt.ylim(0,1200)


plt.xlabel('FLR')
plt.ylabel('Number of Peptides')
plt.title('Comparison of Phospho localization OT on PSMs')
plt.legend(loc='upper left')
plt.grid(True)
